# Step 3 – Leave-one-dataset-out (Section 5.3)

Trains the full pipeline on three cohorts (genes standardised within cohort) and tests on the fourth. Run once per held-out cohort.

*Part of the code accompanying* **Explainable, Leakage-Free Gene Selection for Breast Cancer Histologic Grade** (Lamba, Munjal, Gigras). Run the notebooks from the folder that holds them, with the exported CSV files in `./data`.

**Parameters** – change and re-run for each cohort.

In [ ]:
held_out = "GSE7390"   # one of: GSE7390, UPP, MAINZ, UNT

In [ ]:
import pickle, time, numpy as np, pandas as pd
%run pipeline.ipynb

CG = list(pd.read_csv("./common_genes.csv").iloc[:, 0])
names = ["GSE7390", "UPP", "MAINZ", "UNT"]
data = {c: load(c, genes=CG, zscore=True) for c in names}
out = {}
t0 = time.time()
import os
for held in [held_out]:
    tr = [c for c in names if c != held]
    Xtr = np.vstack([data[c][0].values for c in tr]); ytr = np.concatenate([data[c][1] for c in tr])
    Xte, yte = data[held][0].values, data[held][1]
    r = ges_fit(Xtr, ytr, seed=7)
    Dte = apply_cuts(Xte[:, r["genes"]], r["cuts"])
    res = {}
    for c in ["HWNB", "NB", "RF"]:
        m = build(c).fit(r["Dtr"], r["ytr"], r["nb"], 3)
        res[c] = metrics(yte, m.predict(Dte))
    out[held] = {"n_train": len(ytr), "n_test": len(yte), "genes": [CG[g] for g in r["genes"]],
                 "n_cfs": r["n_cfs"], "n_cons": r["n_cons"], "res": res}
    print(held, len(ytr), r["n_cfs"], r["n_cons"], {c: round(res[c]["macro_f1"], 3) for c in res},
          "%.0fs" % (time.time() - t0), flush=True)
pickle.dump(out, open(f"./lodo_{held}.pkl", "wb"))
print("DONE")